# Lab: Marketing Channel Statistical Analysis
## Parts 2–3 — Pairwise Comparisons & Multiple Corrections

This notebook:
- Compares **CPA** across all channel pairs using **independent t-tests** + Cohen's d
- Compares **conversion rates** across all channel pairs using **Fisher's exact test**
- Applies **Bonferroni** and **Benjamini–Hochberg FDR** corrections
- Visualises significant findings before and after correction

---  

In [ ]:
# ── 0. Imports & load data ───────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import fisher_exact, false_discovery_control
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

df = pd.read_csv('marketing_data.csv', parse_dates=['date'])
CHANNELS = sorted(df['channel'].unique())
ALPHA = 0.05

print(f'Loaded {len(df)} rows | {len(CHANNELS)} channels')
print(f'Channels: {CHANNELS}')

## Step 4 — t-tests on CPA (continuous metric)

**H₀**: Mean daily CPA of channel A = mean daily CPA of channel B  
**H₁**: They differ  
We use an **independent two-sample t-test** (Welch's variant, `equal_var=False`).

In [ ]:
# ── 1. Prep CPA series ───────────────────────────────────────────────────────
daily_cpa = (
    df.dropna(subset=['cpa'])
      .loc[df['cpa'].notna() & np.isfinite(df['cpa'])]
)

def cohens_d(a, b):
    pooled = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    return (np.mean(b) - np.mean(a)) / pooled if pooled > 0 else 0.0

def effect_label(d):
    ad = abs(d)
    if ad < 0.2:  return 'negligible'
    if ad < 0.5:  return 'small'
    if ad < 0.8:  return 'medium'
    return 'large'

cpa_results = []
for ch_a, ch_b in combinations(CHANNELS, 2):
    vals_a = daily_cpa.loc[daily_cpa['channel'] == ch_a, 'cpa'].values
    vals_b = daily_cpa.loc[daily_cpa['channel'] == ch_b, 'cpa'].values
    if len(vals_a) < 3 or len(vals_b) < 3:
        continue
    t_stat, p_val = stats.ttest_ind(vals_a, vals_b, equal_var=False)
    mean_a, mean_b = np.mean(vals_a), np.mean(vals_b)
    diff = mean_b - mean_a
    pct_diff = diff / mean_a * 100 if mean_a != 0 else np.nan
    d = cohens_d(vals_a, vals_b)
    cpa_results.append(dict(
        channel_a=ch_a, channel_b=ch_b,
        mean_a=round(mean_a, 2), mean_b=round(mean_b, 2),
        diff=round(diff, 2), pct_diff=round(pct_diff, 1),
        t_stat=round(t_stat, 3), p_value=round(p_val, 6),
        cohens_d=round(d, 3), effect_size=effect_label(d),
        significant=(p_val < ALPHA),
    ))
    print(f'{ch_a} vs {ch_b}: CPA {mean_a:.1f} vs {mean_b:.1f} | '
          f't={t_stat:.2f} p={p_val:.4f} d={d:.2f} ({effect_label(d)}) '
          f'[{"✓" if p_val<ALPHA else "✗"}]')

cpa_df = pd.DataFrame(cpa_results)
n_cpa = len(cpa_df)
n_sig_cpa = cpa_df['significant'].sum()
print(f'\n{n_cpa} CPA comparisons | {n_sig_cpa} significant at α={ALPHA}')

In [ ]:
# ── 2. CPA p-value heatmap ───────────────────────────────────────────────────
p_matrix = pd.DataFrame(np.ones((len(CHANNELS), len(CHANNELS))),
                         index=CHANNELS, columns=CHANNELS)
for _, row in cpa_df.iterrows():
    p_matrix.loc[row['channel_a'], row['channel_b']] = row['p_value']
    p_matrix.loc[row['channel_b'], row['channel_a']] = row['p_value']

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.eye(len(CHANNELS), dtype=bool)
sns.heatmap(p_matrix, annot=True, fmt='.3f', cmap='RdYlGn_r',
            vmin=0, vmax=0.1, ax=ax, mask=mask,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'p-value (red = significant)'})
np.fill_diagonal(p_matrix.values, np.nan)
ax.set_title('CPA Pairwise t-test p-values\n(red = p < 0.05, green = not significant)', fontsize=12)
plt.tight_layout()
plt.savefig('metric_comparison_heatmap.png', bbox_inches='tight')
plt.show()
print('metric_comparison_heatmap.png saved ✓')

## Step 5 — Fisher's Exact Test on Conversion Rates (binary outcome)

In [ ]:
# ── 3. Aggregate conversion counts per channel ───────────────────────────────
conv_agg = df.groupby('channel').agg(
    conversions=('conversions', 'sum'),
    clicks=('clicks', 'sum'),
).reset_index()
conv_agg['non_conversions'] = conv_agg['clicks'] - conv_agg['conversions']
conv_agg['rate']            = conv_agg['conversions'] / conv_agg['clicks']

print('=== Conversion Counts by Channel ===')
print(conv_agg[['channel','conversions','clicks','non_conversions','rate']]
      .sort_values('rate', ascending=False).to_string(index=False))

In [ ]:
# ── 4. Fisher's exact pairwise tests ─────────────────────────────────────────
conv_lookup = conv_agg.set_index('channel')
fisher_results = []

for ch_a, ch_b in combinations(CHANNELS, 2):
    ca = conv_lookup.loc[ch_a]
    cb = conv_lookup.loc[ch_b]
    table = [
        [int(ca['conversions']), int(ca['non_conversions'])],
        [int(cb['conversions']), int(cb['non_conversions'])],
    ]
    odds_ratio, p_val = fisher_exact(table, alternative='two-sided')
    rate_a, rate_b = ca['rate'], cb['rate']
    diff = rate_b - rate_a
    pct_diff = diff / rate_a * 100 if rate_a != 0 else np.nan
    fisher_results.append(dict(
        channel_a=ch_a, channel_b=ch_b,
        conv_a=int(ca['conversions']), clicks_a=int(ca['clicks']),
        conv_b=int(cb['conversions']), clicks_b=int(cb['clicks']),
        rate_a=round(rate_a, 5), rate_b=round(rate_b, 5),
        diff=round(diff, 5), pct_diff=round(pct_diff, 1),
        odds_ratio=round(odds_ratio, 3), p_value=round(p_val, 8),
        significant=(p_val < ALPHA),
    ))
    print(f'{ch_a} vs {ch_b}: rate {rate_a:.4f} vs {rate_b:.4f} | '
          f'OR={odds_ratio:.2f} p={p_val:.4e} [{"✓" if p_val<ALPHA else "✗"}]')

fisher_df = pd.DataFrame(fisher_results)
n_fisher = len(fisher_df)
n_sig_fisher = fisher_df['significant'].sum()
print(f'\n{n_fisher} Fisher comparisons | {n_sig_fisher} significant at α={ALPHA}')

In [ ]:
# ── 5. Rate comparison bar chart ─────────────────────────────────────────────
rate_plot = conv_agg.sort_values('conversions', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(rate_plot['channel'],
               rate_plot['rate'] * 100,
               color=sns.color_palette('tab10', n_colors=len(CHANNELS)))
for bar, val in zip(bars, rate_plot['rate'] * 100):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=9)
ax.set_xlabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by Channel (clicks → conversions)')
plt.tight_layout()
plt.savefig('rate_comparison.png', bbox_inches='tight')
plt.show()
print('rate_comparison.png saved ✓')

## Step 6 — Multiple Comparisons Correction

With 7 channels we have **21 unique pairs** per metric → 42 total tests.  
At α=0.05 we expect `42 × 0.05 ≈ 2.1` false positives by chance alone.

In [ ]:
# ── 6. Multiple comparisons problem ─────────────────────────────────────────
total_comparisons = n_cpa + n_fisher
expected_fp = total_comparisons * ALPHA
print(f'Total comparisons made:  {total_comparisons} ({n_cpa} CPA + {n_fisher} Fisher)')
print(f'Expected false positives at α={ALPHA}: {expected_fp:.1f}')
print(f'Some of our {cpa_df["significant"].sum() + fisher_df["significant"].sum()} '
      f'"significant" results are likely noise.')

In [ ]:
# ── 7. Bonferroni correction ─────────────────────────────────────────────────
alpha_bonf_cpa    = ALPHA / n_cpa
alpha_bonf_fisher = ALPHA / n_fisher

cpa_df['significant_bonferroni']    = cpa_df['p_value']    < alpha_bonf_cpa
fisher_df['significant_bonferroni'] = fisher_df['p_value'] < alpha_bonf_fisher

print(f'Bonferroni α (CPA):    {alpha_bonf_cpa:.5f}')
print(f'Bonferroni α (Fisher): {alpha_bonf_fisher:.5f}')
print(f'CPA: {cpa_df["significant_bonferroni"].sum()} / {n_cpa} significant after Bonferroni')
print(f'Fisher: {fisher_df["significant_bonferroni"].sum()} / {n_fisher} significant after Bonferroni')

bonf_cpa_sig = cpa_df[cpa_df['significant_bonferroni']]
if len(bonf_cpa_sig):
    print('\nCPA pairs significant after Bonferroni:')
    print(bonf_cpa_sig[['channel_a','channel_b','mean_a','mean_b','p_value','cohens_d']].to_string(index=False))

In [ ]:
# ── 8. Benjamini-Hochberg FDR correction ─────────────────────────────────────
cpa_pvals    = cpa_df['p_value'].values
fisher_pvals = fisher_df['p_value'].values

cpa_adj    = false_discovery_control(cpa_pvals,    method='bh')
fisher_adj = false_discovery_control(fisher_pvals, method='bh')

cpa_df['p_value_fdr']    = cpa_adj.round(6)
cpa_df['significant_fdr'] = cpa_adj < ALPHA

fisher_df['p_value_fdr']    = fisher_adj.round(8)
fisher_df['significant_fdr'] = fisher_adj < ALPHA

print(f'CPA: {cpa_df["significant_fdr"].sum()} / {n_cpa} significant after BH-FDR')
print(f'Fisher: {fisher_df["significant_fdr"].sum()} / {n_fisher} significant after BH-FDR')

fdr_cpa_sig = cpa_df[cpa_df['significant_fdr']]
if len(fdr_cpa_sig):
    print('\nCPA pairs significant after FDR:')
    print(fdr_cpa_sig[['channel_a','channel_b','mean_a','mean_b',
                        'p_value','p_value_fdr','cohens_d','effect_size']].to_string(index=False))

fdr_fisher_sig = fisher_df[fisher_df['significant_fdr']]
if len(fdr_fisher_sig):
    print('\nFisher pairs significant after FDR:')
    print(fdr_fisher_sig[['channel_a','channel_b','rate_a','rate_b',
                           'p_value','p_value_fdr','odds_ratio']].to_string(index=False))

In [ ]:
# ── 9. Summary comparison table & chart ──────────────────────────────────────
summary_data = {
    'Method':       ['Uncorrected (α=0.05)', 'Bonferroni', 'BH-FDR'],
    'CPA sig.':     [
        cpa_df['significant'].sum(),
        cpa_df['significant_bonferroni'].sum(),
        cpa_df['significant_fdr'].sum(),
    ],
    'Fisher sig.':  [
        fisher_df['significant'].sum(),
        fisher_df['significant_bonferroni'].sum(),
        fisher_df['significant_fdr'].sum(),
    ],
}
summary_df = pd.DataFrame(summary_data)
summary_df['Total sig.'] = summary_df['CPA sig.'] + summary_df['Fisher sig.']
print('=== Correction Method Comparison ===')
print(summary_df.to_string(index=False))

# Bar chart
x = np.arange(3)
width = 0.3
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width, summary_df['CPA sig.'],    width, label='CPA (t-test)',   color='steelblue')
ax.bar(x,         summary_df['Fisher sig.'], width, label='Conversion (Fisher)', color='coral')
ax.bar(x + width, summary_df['Total sig.'],  width, label='Total',          color='seagreen', alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(summary_df['Method'])
ax.set_ylabel('Number of Significant Comparisons')
ax.set_title('Effect of Multiple Comparisons Correction\non Significant Results')
ax.legend()
ax.axhline(expected_fp, color='red', linestyle='--', alpha=0.6,
           label=f'Expected FPs by chance ({expected_fp:.1f})')
ax.legend()
plt.tight_layout()
plt.savefig('correction_comparison.png', bbox_inches='tight')
plt.show()
print('correction_comparison.png saved ✓')

# Save results for downstream use
cpa_df.to_csv('cpa_comparisons.csv', index=False)
fisher_df.to_csv('fisher_comparisons.csv', index=False)
print('cpa_comparisons.csv and fisher_comparisons.csv saved ✓')

## Interpretation

| Correction | CPA sig. | Fisher sig. | Interpretation |
|---|---|---|---|
| None (α=0.05) | many | many | Inflated — includes ~2 false positives |
| Bonferroni | fewest | fewest | Very conservative — may miss real effects |
| BH-FDR | moderate | moderate | Best balance for budget decisions |

**Key insight**: Bonferroni is so conservative that we lose statistical power — some real channel differences may be missed. BH-FDR controls the *proportion* of false discoveries rather than eliminating all risk, which is more appropriate when we are comparing many channels and want to identify those worth investigating further.